# Lab 5

**Training a Multimodal Model for Flood Detection with TerraMind**

⏱️ **Estimated Duration:** 90-120 minutes  
📊 **Difficulty Level:** Advanced

📥 **Getting the Lab Materials**

**Getting the Lab Materials:** Clone the repository:
```bash
git clone https://github.com/terrastackai/geospatial-studio.git
cd geospatial-studio/workshop/docs/notebooks
jupyter notebook lab5-flood-multimodal-workflow.ipynb
```

⚠️ **Note:** This lab requires multiple JSON configuration files. Cloning the repository ensures you have all necessary files.

---

## Overview

In this advanced lab, you'll complete an **end-to-end multimodal machine learning workflow** for detecting floods from satellite imagery. This represents a real-world disaster response scenario where you need to:

1. **Onboard multimodal training data** - Upload and configure a dataset with multiple sensor modalities
2. **Register a multimodal foundation model** - Set up TerraMind for fine-tuning
3. **Submit a fine-tuning job** - Train a custom model on your multimodal dataset
4. **Monitor training progress** - Track the training process
5. **Run inference** - Test your trained model on new data
6. **Visualize results** - View predictions in the UI

## What You'll Learn

- How to work with multimodal satellite data (Sentinel-1 SAR + Sentinel-2 optical)
- How to onboard and configure multimodal training datasets
- How to use TerraMind, a multimodal foundation model
- How to configure and submit multimodal fine-tuning jobs
- How to run inference with multimodal models
- Best practices for multimodal geospatial AI

## Prerequisites

### Required Resources

- **GPU Access**: This lab requires GPU resources for model training
- **Training Data**: Sen1Floods11 multimodal dataset (automatically downloaded)
- **Completed Labs**: Labs 1-4 (for SDK familiarity and workflow understanding)

### About the Dataset

We'll use the **Sen1Floods11 dataset**, a multimodal flood detection dataset:

**Sentinel-2 (Optical) Modality:**
- **13 spectral bands** including RGB, NIR, SWIR, and cloud mask
- **10-60m resolution** depending on band
- **Modality tag**: S2L2A

**Sentinel-1 (SAR) Modality:**
- **2 polarization bands**: VV and VH
- **10m resolution**
- **Modality tag**: S1GRD
- **Weather-independent** - works through clouds

**Labels:**
- **Binary classification**: Flood (1) vs No Flood (0)
- **Pixel-level segmentation**

### Why Multimodal?

Combining optical and SAR data provides complementary information:
- **Optical (Sentinel-2)**: Rich spectral information, but affected by clouds
- **SAR (Sentinel-1)**: Penetrates clouds, sensitive to water surfaces
- **Together**: More robust and accurate flood detection

### 💻 CPU Training Support

**This lab uses TerraMind Tiny for CPU compatibility:**

- **TerraMind Tiny** is the smallest version and can run on CPU
- **Training time on CPU**: Approximately 2-4 hours (vs 60-90 minutes on GPU)
- **GPU recommended** but not required for this lab
- If you have a GPU available, training will be significantly faster
- For production workloads, consider using terramind_v1_base or terramind_v1_large with GPU

---

## 1. Setup and Imports

First, let's import the required libraries and set up our connection to Geospatial Studio.

In [1]:
# Import required packages
import json
import time
from IPython.display import display, Markdown

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from geostudio import Client

## 2. Connect to Geospatial Studio

Use the same configuration file from Lab 1:

In [2]:
# Initialize the client
client = Client(geostudio_config_file=".geostudio_config_file")

print("✅ Connected to Geospatial Studio!")

Using api key and base urls from geostudio config file
Using api key and base urls from geostudio config file
Using api key and base urls from geostudio config file
✅ Connected to Geospatial Studio!


## 3. Understanding the Multimodal Workflow

Before we begin, let's understand the complete multimodal workflow:

```mermaid
graph LR
    A[Multimodal Foundation Model<br/>TerraMind] --> D[Fine-tuning Job]
    B[Multimodal Training Dataset<br/>Sentinel-1 + Sentinel-2] --> D
    C[TerraMind Task Template] --> D
    D --> E[Trained Multimodal Model]
    E --> F[Multimodal Inference]
    F --> G[Flood Detection Results]
    
    style A fill:#1f77b4,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#1f77b4,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#1f77b4,stroke:#333,stroke-width:2px,color:#fff
    style D fill:#ff7f0e,stroke:#333,stroke-width:2px,color:#fff
    style E fill:#2ca02c,stroke:#333,stroke-width:2px,color:#fff
    style F fill:#d62728,stroke:#333,stroke-width:2px,color:#fff
    style G fill:#9467bd,stroke:#333,stroke-width:2px,color:#fff
```

**Key Components:**

1. **Multimodal Foundation Model (TerraMind)**: Pre-trained model that understands multiple sensor modalities
2. **Multimodal Training Dataset**: Labeled data with both Sentinel-1 SAR and Sentinel-2 optical imagery
3. **TerraMind Task Template**: Configuration for multimodal segmentation
4. **Fine-tuning Job**: Training process that customizes the model for flood detection
5. **Trained Model**: Your custom multimodal flood detection model
6. **Multimodal Inference**: Running predictions using both sensor types

---

## 4. Step 1: Register the TerraMind Foundation Model

First, we need to register the **TerraMind** multimodal foundation model as our base model.

**About TerraMind:**
- **Multimodal architecture**: Designed to process multiple sensor types simultaneously
- **Pre-trained**: On diverse geospatial datasets with multiple modalities
- **Versions**: Available in tiny, base, and large sizes
- **Fusion strategy**: Learns optimal ways to combine information from different sensors

For this lab, we'll use the **terramind_v1_tiny** model, which is the smallest version and can run on CPU, making it accessible for systems without GPU resources.

In [3]:
# Load the TerraMind tiny backbone configuration
with open("../../../populate-studio/payloads/backbones/backbone-terramind_tiny.json", "r") as f:
    backbone = json.load(f)

print("📋 TerraMind Backbone Configuration:")
print(json.dumps(backbone, indent=2))

📋 TerraMind Backbone Configuration:
{
  "name": "terramind_v1_tiny",
  "description": "Terramind Multimodal tiny backbone base",
  "checkpoint_filename": "terramind_v1_tiny/terramind_v1_tiny.pt",
  "model_params": {
    "backbone": "terramind_v1_tiny",
    "model_category": "terramind"
  }
}


In [4]:
# Register the backbone model
onboard_backbone_response = client.create_base_model(backbone)

print("\n✅ TerraMind foundation model registered successfully!")
print(json.dumps(onboard_backbone_response, indent=2))

# Save the base model ID for later use
base_model_id = onboard_backbone_response['id']
print(f"\n📌 Base Model ID: {base_model_id}")


✅ TerraMind foundation model registered successfully!
{
  "id": "235d4756-4afc-487f-8cd5-5714f70f2d18"
}

📌 Base Model ID: 235d4756-4afc-487f-8cd5-5714f70f2d18


## 5. Step 2: Onboard the Multimodal Training Dataset

Now we'll onboard the Sen1Floods11 multimodal dataset. This process:

1. **Downloads** the dataset from a remote URL
2. **Validates** both modalities and their alignment
3. **Indexes** the multimodal dataset for training
4. **Stores** metadata in the Studio catalog

**Dataset Configuration:**

**Sentinel-2 (S2L2A) Modality:**
- **13 bands**: B01-B12 + Cloud mask (CLD)
- **File suffix**: `_S2Hand.tif`
- **RGB bands**: B02 (Blue), B03 (Green), B04 (Red)

**Sentinel-1 (S1GRD) Modality:**
- **2 bands**: VV and VH polarizations
- **File suffix**: `_S1Hand.tif`
- **Aligned dates**: Temporal alignment with Sentinel-2

**Labels:**
- **2 categories**: No Floods (0), Floods (1)
- **File suffix**: `_LabelHand.tif`

⚠️ **Note**: This process may take several minutes depending on network speed and cluster resources.

In [5]:
# Load the multimodal dataset configuration
with open("../../../populate-studio/payloads/datasets/dataset-flooding_multimodal.json", "r") as f:
    flood_dataset = json.load(f)

print("📋 Multimodal Dataset Configuration:")
print(json.dumps(flood_dataset, indent=2))

📋 Multimodal Dataset Configuration:
{
  "dataset_name": "Sentinel Flood Multimodal Test",
  "data_sources": [
    {
      "bands": [
        {
          "index": "0",
          "band_name": "B01",
          "description": "",
          "scaling_factor": "1"
        },
        {
          "index": "1",
          "band_name": "B02",
          "RGB_band": "B",
          "description": "",
          "scaling_factor": "1"
        },
        {
          "index": "2",
          "band_name": "B03",
          "RGB_band": "G",
          "description": "",
          "scaling_factor": "1"
        },
        {
          "index": "3",
          "band_name": "B04",
          "RGB_band": "R",
          "description": "",
          "scaling_factor": "1"
        },
        {
          "index": "4",
          "band_name": "B05",
          "description": "",
          "scaling_factor": "1"
        },
        {
          "index": "5",
          "band_name": "B06",
          "description": "",
          "sc

In [6]:
# Onboard the multimodal dataset
print("🚀 Starting multimodal dataset onboarding...")
print("This may take several minutes to download and process both modalities.\n")

onboard_dataset_response = client.onboard_dataset(data=flood_dataset)

print("✅ Multimodal dataset onboarding initiated!")
print(json.dumps(onboard_dataset_response, indent=2))

# Save the dataset ID for later use
dataset_id = onboard_dataset_response['dataset_id']
print(f"\n📌 Dataset ID: {dataset_id}")

🚀 Starting multimodal dataset onboarding...
This may take several minutes to download and process both modalities.

✅ Multimodal dataset onboarding initiated!
{
  "Dataset": "submitting - adding dataset and labels",
  "dataset_id": "geodata-m6mjazvh82c5spfqzzvqwq",
  "dataset_url": "https://geospatial-studio-example-data.s3.us-east.cloud-object-storage.appdomain.cloud/sen1floods11_v1.1.4_zip.zip"
}

📌 Dataset ID: geodata-m6mjazvh82c5spfqzzvqwq


### Monitor Dataset Onboarding

The onboarding process runs as a background job. We'll poll the status until completion.

**What's happening:**
- Downloading multimodal dataset from remote storage
- Extracting and validating files for both modalities
- Checking temporal alignment between Sentinel-1 and Sentinel-2
- Creating dataset index
- Generating metadata

**Expected statuses:**
- `PENDING`: Job queued
- `RUNNING`: Processing data
- `COMPLETED`: Successfully onboarded
- `FAILED`: Error occurred (check logs)

In [8]:
# Poll dataset onboarding status
print("⏳ Monitoring dataset onboarding progress...\n")

max_attempts = 60  # 30 minutes with 30-second intervals
attempt = 0

while attempt < max_attempts:
    status_response = client.poll_onboard_dataset_until_finished(dataset_id)
    status = status_response.get('status', 'UNKNOWN')
    
    print(f"Attempt {attempt + 1}/{max_attempts} - Status: {status}")
    
    if status in ['COMPLETED', 'Succeeded']:
        print("\n✅ Dataset onboarding completed successfully!")
        print(json.dumps(status_response, indent=2))
        break
    elif status in ['FAILED', 'Failed']:
        print("\n❌ Dataset onboarding failed!")
        print(json.dumps(status_response, indent=2))
        break
    
    attempt += 1
    time.sleep(30)  # Wait 30 seconds before next check

if attempt >= max_attempts:
    print("\n⚠️ Timeout: Dataset onboarding is still in progress.")
    print("Check the status manually using: client.get_dataset_status(dataset_id)")

⏳ Monitoring dataset onboarding progress...

Succeeded - 421 secondss
Attempt 1/60 - Status: Succeeded
Succeeded - 451 seconds
Attempt 2/60 - Status: Succeeded


KeyboardInterrupt: 

## 6. Step 3: Register the TerraMind Task Template

Now we need to register the task template that defines how to train with TerraMind.

The TerraMind segmentation template is specifically designed for multimodal training and includes:
- Multimodal fusion configuration
- Decoder architecture settings
- Training hyperparameters optimized for TerraMind

In [9]:
# Load the TerraMind segmentation template
with open("../../../populate-studio/payloads/templates/template-terramind_seg.json", "r") as f:
    terramind_template = json.load(f)

print("📋 TerraMind Segmentation Template:")
print(f"Name: {terramind_template['name']}")
print(f"Description: {terramind_template['description']}")
print(f"Purpose: {terramind_template['purpose']}")
print(f"Model Category: {terramind_template['extra_info']['model_category']}")

📋 TerraMind Segmentation Template:
Name: terramind Segmentation
Description: Terramind multimodal task for Segmantation
Purpose: Segmentation
Model Category: terramind


In [10]:
# Register the template
template_response = client.create_task(terramind_template)

print("\n✅ TerraMind template registered successfully!")
print(json.dumps(template_response, indent=2))

# Save the template ID for later use
tune_template_id = template_response['id']
print(f"\n📌 Template ID: {tune_template_id}")


✅ TerraMind template registered successfully!
{
  "id": "21932c6c-62df-408e-ba87-3f0121d29a09"
}

📌 Template ID: 21932c6c-62df-408e-ba87-3f0121d29a09


## 7. Step 4: Submit the Fine-Tuning Job

Now we'll submit a fine-tuning job to train our multimodal flood detection model.

**Training Configuration:**
- **Base Model**: TerraMind v1 Tiny (CPU-compatible)
- **Dataset**: Sen1Floods11 multimodal
- **Task**: Segmentation (pixel-level flood classification)
- **Epochs**: 10 (can be adjusted)
- **Learning Rate**: 6e-5 (optimized for TerraMind)
- **Batch Size**: 2 (suitable for CPU training)

### 💻 CPU vs GPU Training

**CPU Training (TerraMind Tiny):**
- ✅ No GPU required
- ✅ Works on any system
- ⏱️ Slower: 2-4 hours
- 📊 Good for learning and testing

**GPU Training (Optional):**
- ⚡ Much faster: 60-90 minutes
- 🎯 Better for production
- 💰 Requires GPU resources

**Note:** If you have a GPU available, the same code will automatically use it for faster training.

In [11]:
# Define the fine-tuning payload
tune_payload = {
    "name": "flood-multimodal-terramind",
    "description": "Multimodal flood detection model trained with TerraMind on Sen1Floods11",
    "dataset_id": dataset_id,
    "base_model_id": base_model_id,
    "tune_template_id": tune_template_id
}

print("📋 Fine-tuning Configuration:")
print(json.dumps(tune_payload, indent=2))

📋 Fine-tuning Configuration:
{
  "name": "flood-multimodal-terramind",
  "description": "Multimodal flood detection model trained with TerraMind on Sen1Floods11",
  "dataset_id": "geodata-m6mjazvh82c5spfqzzvqwq",
  "base_model_id": "235d4756-4afc-487f-8cd5-5714f70f2d18",
  "tune_template_id": "21932c6c-62df-408e-ba87-3f0121d29a09"
}


In [12]:
# Submit the fine-tuning job
print("🚀 Submitting multimodal fine-tuning job...\n")

tune_response = client.submit_tune(tune_payload, output='json')

print("✅ Fine-tuning job submitted successfully!")
print(json.dumps(tune_response, indent=2))

# Save the tune ID for later use
tune_id = tune_response['tune_id']
print(f"\n📌 Tune ID: {tune_id}")
print(f"\n💡 Training will take approximately 60-90 minutes.")
print(f"You can monitor progress in the MLflow UI or Studio UI.")

🚀 Submitting multimodal fine-tuning job...

✅ Fine-tuning job submitted successfully!
{
  "tune_id": "geotune-v7zbfsjkmmszbiqtkjlwsy",
  "mcad_id": "kjob-geotune-v7zbfsjkmmszbiqtkjlwsy",
  "status": "In_progress",
  "message": null
}

📌 Tune ID: geotune-v7zbfsjkmmszbiqtkjlwsy

💡 Training will take approximately 60-90 minutes.
You can monitor progress in the MLflow UI or Studio UI.


## 8. Step 5: Monitor Training Progress

Training a multimodal model takes time. We'll poll the status until completion.

**What's happening during training:**
1. **Initialization**: Loading TerraMind base model and multimodal dataset
2. **Data Loading**: Processing both Sentinel-1 and Sentinel-2 modalities
3. **Training**: Iterating through epochs, learning multimodal fusion
4. **Validation**: Evaluating performance on validation set
5. **Checkpointing**: Saving model state periodically
6. **Completion**: Finalizing and storing the trained model

**Expected Duration:**
- **CPU (TerraMind Tiny)**: 2-4 hours
- **GPU (V100/A100)**: 60-90 minutes
- **GPU (Older)**: 90-120 minutes

**Note:** This lab uses TerraMind Tiny which is optimized for CPU training. Training will complete successfully on CPU, just takes longer.

In [13]:
# Poll training status
print("⏳ Monitoring training progress...\n")
print("This will take approximately 60-90 minutes.")
print("You can also monitor in the Studio UI or MLflow.\n")

client.poll_finetuning_until_finished(tune_id=tune_id)

print("\n✅ Training completed successfully!")
print(f"Your trained model '{tune_payload['name']}' is now available in the Model Catalog.")

⏳ Monitoring training progress...

This will take approximately 60-90 minutes.
You can also monitor in the Studio UI or MLflow.

Finished - Epoch: None - 426 secondsnds

✅ Training completed successfully!
Your trained model 'flood-multimodal-terramind' is now available in the Model Catalog.


## 9. Step 6: Run Multimodal Inference

Now let's test our trained multimodal model on new data!

**Important**: For multimodal inference, you need a location where BOTH Sentinel-1 and Sentinel-2 data are available.

For this example, we'll configure an inference request. You'll need to provide a test area with both sensor coverages.

In [ ]:
# Define the multimodal inference payload
# Note: Adjust bbox and dates for your specific test area
inference_payload = {
    "model_display_name": "flood-multimodal-demo",
    "location": "Flood Test Area",
    "description": "Multimodal flood detection test",
    "spatial_domain": {
        "bbox": [],  # Add bbox: [min_lon, min_lat, max_lon, max_lat]
        "urls": [],
        "tiles": [],
        "polygons": []
    },
    "temporal_domain": ["2024-01-15"],  # Adjust date
    "pipeline_steps": [
        {"status": "READY", "process_id": "sentinelhub-connector", "step_number": 0},
        {"status": "WAITING", "process_id": "terratorch-inference", "step_number": 1},
        {"status": "WAITING", "process_id": "postprocess-generic", "step_number": 2},
        {"status": "WAITING", "process_id": "push-to-geoserver", "step_number": 3}
    ],
    "model_input_data_spec": [
        {
            "bands": [
                {"index": "0", "band_name": "B01", "scaling_factor": "1"},
                {"index": "1", "band_name": "B02", "RGB_band": "B", "scaling_factor": "1"},
                {"index": "2", "band_name": "B03", "RGB_band": "G", "scaling_factor": "1"},
                {"index": "3", "band_name": "B04", "RGB_band": "R", "scaling_factor": "1"},
                {"index": "4", "band_name": "B05", "scaling_factor": "1"},
                {"index": "5", "band_name": "B06", "scaling_factor": "1"},
                {"index": "6", "band_name": "B07", "scaling_factor": "1"},
                {"index": "7", "band_name": "B08", "scaling_factor": "1"},
                {"index": "8", "band_name": "B8A", "scaling_factor": "1"},
                {"index": "9", "band_name": "B09", "scaling_factor": "1"},
                {"index": "10", "band_name": "B11", "scaling_factor": "1"},
                {"index": "11", "band_name": "B12", "scaling_factor": "1"},
                {"index": "12", "band_name": "CLD", "scaling_factor": "1"}
            ],
            "connector": "sentinelhub",
            "collection": "s2_l2a",
            "modality_tag": "S2L2A"
        },
        {
            "bands": [
                {"index": "0", "band_name": "VV"},
                {"index": "1", "band_name": "VH"}
            ],
            "connector": "sentinelhub",
            "collection": "s1_grd",
            "modality_tag": "S1GRD",
            "align_dates": "true"
        }
    ],
    "geoserver_push": [
        {
            "z_index": 0,
            "workspace": "geofm",
            "layer_name": "input_rgb",
            "display_name": "Input RGB",
            "filepath_key": "model_input_original_image_rgb",
            "geoserver_style": {"rgb": [
                {"label": "Red", "channel": 1, "maxValue": 255, "minValue": 0},
                {"label": "Green", "channel": 2, "maxValue": 255, "minValue": 0},
                {"label": "Blue", "channel": 3, "maxValue": 255, "minValue": 0}
            ]},
            "visible_by_default": "True"
        },
        {
            "z_index": 1,
            "workspace": "geofm",
            "layer_name": "pred",
            "display_name": "Flood Prediction",
            "filepath_key": "model_output_image",
            "geoserver_style": {"segmentation": [
                {"color": "#000000", "label": "no-flood", "opacity": 0, "quantity": "0"},
                {"color": "#0080FF", "label": "flood", "opacity": 0.7, "quantity": "1"}
            ]},
            "visible_by_default": "True"
        }
    ]
}

print("📋 Inference Configuration Ready")
print(f"Modalities: S2L2A + S1GRD")

In [ ]:
# Submit the inference request
print("🚀 Submitting multimodal inference...\n")

inference_response = client.try_out_tune(tune_id=tune_id, data=inference_payload)

print("✅ Inference submitted successfully!")
print(json.dumps(inference_response, indent=2))

print("\n💡 View results in the Studio UI under Model & Tunes > History")

## 10. Viewing Results

Your multimodal flood detection results are now available in the UI!

**To view:**
1. Navigate to Model Catalog in Studio UI
2. Find "flood-multimodal-terramind"
3. Click History tab
4. View your inference on the map

**What to observe:**
- Blue overlay shows detected floods
- Compare with RGB input
- Notice SAR helps through clouds
- Assess multimodal fusion benefits

## 11. Conclusion

Congratulations! You've successfully set up a multimodal flood detection workflow using TerraMind.

### What You've Accomplished

1. **Registered TerraMind**: Set up a multimodal foundation model
2. **Onboarded Multimodal Data**: Configured Sen1Floods11 dataset with Sentinel-1 and Sentinel-2
3. **Understanding**: Learned about multimodal fusion and its benefits

### Next Steps

To complete the full workflow:
1. Load and register the TerraMind segmentation template
2. Submit a fine-tuning job with your multimodal dataset
3. Monitor training progress (60-90 minutes)
4. Run inference on new multimodal data
5. Visualize and evaluate results

### Key Takeaways

- **Multimodal Advantage**: Combining SAR and optical data provides more robust predictions
- **TerraMind**: Specialized architecture for processing multiple sensor types
- **Data Alignment**: Critical to ensure temporal alignment between modalities
- **Applications**: Flood detection, disaster response, climate monitoring

### Resources

- **Documentation**: [Geospatial Studio Docs](https://terrastackai.github.io/geospatial-studio/)
- **SDK Reference**: [Geospatial Studio SDK](https://terrastackai.github.io/geospatial-studio-toolkit/)
- **Dataset**: [Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11)
- **Models**: [IBM Geospatial Models](https://huggingface.co/ibm-nasa-geospatial)

---

## 🎓 Lab Complete!

Thank you for completing Lab 5! You've learned how to work with multimodal geospatial data and TerraMind foundation models.

**Questions or Feedback?**
- Open an issue on [GitHub](https://github.com/terrastackai/geospatial-studio)
- Join our community discussions

Happy mapping! 🌍🛰️🌊